# Intro

Recoding of https://github.com/Daniel-EST/deep-steganography/ but in pytorch while adding below to fit paper being audited as code was not available to test

from paper: 500 training images, 50 validation images, and 50 test images, all uniformly resized to a manageable dimension of 64x64x3 pixels

text > Huffman > LSB > NN Embed > NN decode > LSB > Huffman > text

LSB - use pillow library (https://pypi.org/project/pillow/)

Huffman - use geeks for geeks

In [45]:
from LSBSteg.LSBSteg import LSBSteg
from dahuffman import HuffmanCodec

# code example from https://github.com/RobinDavid/LSB-Steganography libraru, lsb is using
# LSB 
def LSBEmbed(text,image):
    # Embed text into image using least sig bit method
    steg = LSBSteg(image)
    img_encoded = steg.encode_binary(text)
    return img_encoded

def LSBExtract(image):
    # pull least sig bit from image and reformat into embedded text
    steg = LSBSteg(image)
    extracted_text = steg.decode_binary()
    return extracted_text

# huffman serialization assisted with chatgpt for debugging, dahuffman use from PyPl example
def HuffmanEncode(data):
    # encode text into huffman tree + map
    codec = HuffmanCodec.from_data(data)
    encoded_data = codec.encode(data)

    # finally prepend all so tree:data is one block
    return codec, encoded_data

def HuffmanDecode(codec,data):
    # using Huffman map reconstruct text to original values and decode / unzip data
    decoded_data = codec.decode(data)
    return decoded_data

In [46]:
from datasets import load_dataset
import os, certifi
import numpy as np
import random

os.environ['SSL_CERT_FILE'] = certifi.where()
def is_rgb(example):
    return example['image'].mode == 'RGB'
# option 1 uses "Hello World!" for all to reuse codec and make things simpler
temp_data = load_dataset("zh-plus/tiny-imagenet")
tiny_imageNet_rgb = temp_data.filter(is_rgb)

train_pool = tiny_imageNet_rgb['train']
test_pool = tiny_imageNet_rgb['valid']

ex_data = "Hello World!"
codec,encoded_data = HuffmanEncode(ex_data)

# 2. Sample from Training Pool
num_train = 500
train_indices = random.sample(range(len(train_pool)), num_train * 2)
c_train_imgs = [train_pool[i]['image'] for i in train_indices[:num_train]]
s_train_imgs = [train_pool[i]['image'] for i in train_indices[num_train:]]

# 3. Sample from Test Pool (Audit Set)
num_test = 50
test_indices = random.sample(range(len(test_pool)), num_test * 2)
c_test_imgs = [test_pool[i]['image'] for i in test_indices[:num_test]]
s_test_imgs = [test_pool[i]['image'] for i in test_indices[num_test:]]

s_train_steg = []
for img in s_train_imgs:
    np_img = np.array(img)
    s_train_steg.append( LSBEmbed(encoded_data,np_img))

s_test_steg = []
for img in s_test_imgs:
    np_img = np.array(img)
    s_test_steg.append( LSBEmbed(encoded_data,np_img))

print(f"Pools Ready: {len(c_train_imgs)} Train pairs, {len(c_test_imgs)} Test pairs.")

Pools Ready: 500 Train pairs, 50 Test pairs.


In [ ]:
# steg NN


In [ ]:
# --- Training Configuration ---


print("Training Complete.")

Starting Training on 500 pairs...
Epoch [1/10] | Total Loss: 0.074884 | Cover MSE: 0.031396 | Secret MSE: 0.004352


KeyboardInterrupt: 

In [58]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

print(torch.cuda.is_available())

def extract_with_transform(tensor):
    # 1. Clean the tensor (Detach and Squeeze)
    # ToPILImage does not handle 4D batches automatically
    t = tensor.detach().cpu().squeeze(0) if tensor.dim() == 4 else tensor.detach().cpu()

    # 2. CRITICAL STEP: Manual Rounding
    # We round while still a tensor to ensure 127.9 becomes 128.0
    t = torch.clamp(t * 255.0, 0, 255).round() / 255.0

    # 3. Use Transform for the rest
    # This handles the [C,H,W] -> [H,W,C] swap and uint8 conversion
    to_img = transforms.ToPILImage()
    pil_img = to_img(t)
    
    # Return as numpy array for your LSBSteg library
    return np.array(pil_img)

def calculate_metrics(original, reconstructed):
    # Ensure images are uint8 and same shape
    original = original.astype(np.uint8)
    reconstructed = reconstructed.astype(np.uint8)
    
    psnr_val = psnr(original, reconstructed)
    # For SSIM on color images, set channel_axis=-1 (HWC format)
    ssim_val = ssim(original, reconstructed, channel_axis=-1)
    
    return psnr_val, ssim_val

i=0
for i in range(i,10):
    secret_input = steg_transform(s_test_steg[i]).unsqueeze(0).to(device)
    cover_input = img_transform(c_test_imgs[i]).unsqueeze(0).to(device)

    # 2. Model Inference
    model.eval()
    with torch.no_grad():
        container, revealed = model(secret_input, cover_input)
    rev_img = extract_from_tensor(revealed)
    sec_img = extract_from_tensor(secret_input)
    psnr_val, ssim_val = calculate_metrics(sec_img,rev_img)
    # print(sec_img,rev_img,'\n\n\n')
    print(f'PSNR: {psnr_val}, SSIM: {ssim_val}')
    # rev_img = extract_from_tensor(rev_img)
    # print(rev_img)
    extracted_code = LSBExtract(rev_img)
    print(extracted_code)
    steg_text = HuffmanDecode(codec,extracted_code)

    print(f"steg text for image {i}: {steg_text}")

False
PSNR: 23.04843676466147, SSIM: 0.8719628100364026


SteganographyException: No available slot remaining (image filled)